# MNIST digit classification

This notebook demonstrates a simple supervised classification task, and shows how we can train a model for the task in a neurosymbolic way. The goal is not to show a *better* approach or an improved model. In fact, we will see that the neurosymbolic loss is identical to the standard cross entropy loss.

By the end of this notebook, we will have seen:

1. How to define predicates associated with model predictions
2. How to construct a loss function from such a predicate
3. How to evaluate the consistency of models, and the relationship bet 


## Preliminaries

To get started, let us first get the standard preambles out of the way. We need `torchvision`, and a bunch of imports.

In [3]:
%load_ext autoreload

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

## The model

Now, we can define a neural network for the standard digit classification task. We will define a simple multilayer perceptron. There is nothing fancy about this network. It will get the job done.

In [5]:
class MNISTClassifier(nn.Module):
    def __init__(self, hidden_size: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(784, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 128)
        self.fc3 = nn.Linear(128, 10)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)  # logits

    def predict_probs(self, x: torch.Tensor) -> torch.Tensor:
        return F.softmax(self.forward(x), dim=-1)

## Loading the dataset

Next, let us set up the data loading infrastructure.

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.Lambda(lambda x: x.view(-1))  # Flatten to 784
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)


Training samples: 60000
Test samples: 10000


## The evaluator

Let us implement a simple evaluator that we can use to assess the quality of a model. 

In [7]:

def evaluate(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += len(y)
    return correct / total

## Supervised learning in two ways

Now, we can implement two different supervised learners. First, we will implement a standard pytorch trainer. Our goal is to keep things as simple as possible. 

In [8]:
def train_standard(n_epochs: int = 20, lr: float = 0.001):
    model = MNISTClassifier()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    for epoch in range(n_epochs):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            acc = evaluate(model, test_loader)
            print(f"Epoch {epoch + 1:2d}: test accuracy = {acc:.4f}")

    return model, evaluate(model, test_loader)

Next, we can define a new trainer that uses an idea called "logic-as-loss". 

First, we need to define a predicate that says "The label for an image X is the digit Y." We will cal it `Digit(X, Y)`, which takes the value `true` to indicate the decision that the label for `X` is `Y`, and `false` otherwise. To get started, we will first import `pysignet` and declare the predicate symbol and its two arguments.

In [9]:
import pysignet as psn 

Digit = psn.Symbol("Digit")
X, Y = psn.Variable("X Y")

Next, we can define the expression we care about. In our case it is quite simple. Our training data asserts that for each training image `X`, the label is the ground truth label `Y`. To capture this, we just need to ensure that the truth of the expression `Digit(X, Y)` is made as high as possible for every pair `X` and `Y` in the training data. 

In [10]:
expr = Digit(X, Y)

Now we can write the training loop. It is similar to the above training loop, with only three differences.

1. We need to connect the name `Digit` to the model we want to train
2. We need to compile the expression whose truth we want to maximize into a loss generator
3. In the training loop, we need to map the inputs `X` and outputs `Y` in the complied version of the expression `Digit(X, Y)` to the current batch we have

These differences are annotated in the code below.

Note: The learning rate is also scaled by the batch size. This is technically not necessary, but is present to ensure that the two runs (supervised and logic-to-loss) are mathematically identical. The loss is averaged over the batch when we call `F.cross_entropy`, and is summed over the batch with the default parameters for logic-to-loss. So we can scale the learning rate accordingly for stochastic gradient descent to be identical. This scaling is more messy for Adam, so in practice, when we use Adam, we will likely see small differences.

In [11]:

def train_neurosymbolc(n_epochs: int = 20, lr: float = 0.001/32):
    model = MNISTClassifier()

    # 1. A mapping that says that the Digit predicate is instantiated with the 
    # model we have.
    predicates = {"Digit": model}

    # 2. Compile the expression to a loss genreator
    loss_generator  = psn.logic_to_loss(expr, predicates)

    optimizer = torch.optim.SGD(loss_generator.trainable_parameters, lr=lr)

    for epoch in range(n_epochs):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()

            # 3. Compute the loss for this particular batch of examples
            loss = loss_generator.loss(X=x, Y=y)
            
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            acc = evaluate(model, test_loader)
            print(f"Epoch {epoch + 1:2d}: test accuracy = {acc:.4f}")

    return model, evaluate(model, test_loader)

## Evaluating the two training approaches

Let us take the two trainers for a spin. First, the standard approach.

In [12]:
torch.manual_seed(1)

model, accuracy = train_standard()

Epoch 10: test accuracy = 0.9274
Epoch 20: test accuracy = 0.9457


We get a model that is about 97.5% accurate. This is not the best we can do for this task, even with this model. But our goal at this point is not to match the standard published results. Our goal is to see if the above numbers can be matched by the neurosymbolic trainer.

In [13]:
torch.manual_seed(1)

model, accuracy = train_neurosymbolc()

Epoch 10: test accuracy = 0.9274
Epoch 20: test accuracy = 0.9457


We see that the two models obtain essentially the same accuracy. There may be minor differences because of numeric reasons, rather than anything different in the math underlying the two runs.

If all we need to do is train a supervised model, then there is no real advantage of the above approach. But thinking of model predictions in terms of logic opens the doors to other forms of supervision. Other notebooks explore these ideas.

## Evaluating consistency of models

Above, we measured the accuracy of models for evaluation. Accuracy is essentially asking the question 
> *How often does the model prediction match the ground truth label?*

We could ask the same question in another way: 

> *How often is the following statement `false`: "The label for an image X is the digit Y."*

The statement is exactly the expression `Digit(X, Y)`. So instead of measuring accuracy of a model, we could equivalently measure the consistency of its decisions with respect to an expression. 

Let us instantiate the idea for our trained model.

In [14]:
evaluator = psn.consistency_report(expr, predicates = {"Digit": model})

for x,y in test_loader:
    evaluator.eval(X=x, Y=y)

In [15]:
evaluator

ConsistencyReport(9457/10000 satisfied)
  global_violation (rho): 0.0543
  global_consistency:     0.9457

The global consistency of the model is simply the fraction of the examples where the model satisfies the constraints. Unsurprisingly, it is identical to the accuracy above. Global violation ($\rho$) is the fraction of examples where the constraints are violated.